<a href="https://colab.research.google.com/github/Swikriti07/Statistics-Essentials-for-Machine-Learning/blob/main/Week2_Nepal_Tourism_Relationship_Detective_Student_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 2 Workshop: Relationship Detective in Nepal Tourism

**Topic:** Bivariate relationships, correlation, feature-target relationships, feature-feature relationships, multicollinearity, VIF, and model-readiness decisions.

## Business context
A Nepal-based tourism company offers trekking and cultural packages across Everest, Annapurna, Langtang, Mustang, Manaslu, and Kathmandu-Pokhara routes. The company wants to understand how booking variables relate to one another before using the data for pricing, customer segmentation, and future machine learning models.

## Your final output
Complete the notebook by running the code cells, inserting your interpretations in the answer cells

## Learning outcomes
By the end of this notebook, you should be able to:

1. Explore relationships between two variables using visualizations.
2. Distinguish feature-target relationships from feature-feature relationships.
3. Interpret a correlation matrix and heatmap.
4. Identify possible multicollinearity among predictors.
5. Use VIF conceptually and computationally.
6. Make feature decisions for a machine learning problem using both statistics and domain knowledge.

## Dataset variables

The dataset contains tourism booking records. Important variables include:

- `TotalSpendUSD`: total booking value.
- `SpendPerPersonUSD`: package spend per traveler.
- `TripDays`: itinerary length.
- `MaxAltitudeM`: approximate route altitude.
- `DifficultyScore`: route difficulty from 1 to 5.
- `PermitComplexityScore`: complexity of permits/logistics.
- `Travelers`: group size.
- `LeadDays`: booking lead time.
- `SatisfactionScore`: post-trip satisfaction from 1 to 5.
- `HighValueBooking`: whether the booking value is in the top quartile.

### Key idea for this workshop
 We are asking:

> Which variables move together, and what does that mean for modelling or business decisions?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', 100)

DATA_PATH = '/content/Week2_Nepal_Tourism_Relationships_Dataset.xlsx'
df = pd.read_excel(DATA_PATH, sheet_name='Tourism_Bookings')

df.head()

In [ ]:
# Basic structure of the dataset
print(df.shape)
df.info()

In [ ]:
# Quick descriptive statistics for numerical variables
num_cols = df.select_dtypes(include='number').columns
round(df[num_cols].describe(), 2)

---
# Task 1: Domain intuition before analysis

Before making graphs, use your domain knowledge.

## Question 1
Choose **three pairs of variables** that you expect to be related in Nepal tourism bookings. For each pair, briefly explain the expected relationship.

Example: `TripDays` and `SpendPerPersonUSD` may be positively related because longer itineraries usually cost more.

### Your answer
1.  
2.  
3.

---
# Task 2: Distribution check

 Distribution still matters to explore relationships . Skewed variables and extreme values can affect correlation.

In [ ]:
# Histograms of key numerical variables
plot_cols = ['Travelers', 'LeadDays', 'TripDays', 'MaxAltitudeM', 'SpendPerPersonUSD', 'TotalSpendUSD', 'SatisfactionScore']

df[plot_cols].hist(figsize=(14, 9), bins=20)
plt.suptitle('Distributions of Key Tourism Booking Variables', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots for variables likely to have spread or outliers
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(y=df['TotalSpendUSD'], ax=axes[0])
axes[0].set_title('Total Spend')
sns.boxplot(y=df['SpendPerPersonUSD'], ax=axes[1])
axes[1].set_title('Spend Per Person')
sns.boxplot(y=df['LeadDays'], ax=axes[2])
axes[2].set_title('Lead Days')
plt.tight_layout()
plt.show()

## Question 2
Based on the histograms and box plots:

- Which variables look most skewed?
- Which variables may influence correlation strongly because of their spread?
- Why should we look at distributions before interpreting relationships?

### Your answer

---
# Task 3: Bivariate relationship exploration

In this section, you will explore how two variables behave together.

Look for:

- direction: positive, negative, or no clear trend
- strength: strong, moderate, weak
- shape: linear or nonlinear
- clusters: different groups or route types
- business interpretation

In [ ]:
# Relationship 1: Trip length and spend per person
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='TripDays', y='SpendPerPersonUSD', hue='Region', alpha=0.8)
sns.regplot(data=df, x='TripDays', y='SpendPerPersonUSD', scatter=False, color='black')
plt.title('Trip Days vs Spend Per Person')
plt.show()

In [ ]:
# Relationship 2: Altitude and difficulty
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='MaxAltitudeM', y='DifficultyScore', hue='RouteName', legend=False, alpha=0.8)
plt.title('Maximum Altitude vs Difficulty Score')
plt.show()

In [ ]:
# Relationship 3: Travelers and total spend
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='Travelers', y='TotalSpendUSD', hue='MarketSegment', alpha=0.8)
sns.regplot(data=df, x='Travelers', y='TotalSpendUSD', scatter=False, color='black')
plt.title('Number of Travelers vs Total Booking Spend')
plt.show()

In [ ]:
# Relationship 4: Lead days by market segment
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='MarketSegment', y='LeadDays')
plt.title('Booking Lead Days by Market Segment')
plt.show()

In [ ]:
# Relationship 5: Satisfaction by accommodation level
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='AccommodationLevel', y='SatisfactionScore', order=['Budget','Standard','Comfort'])
plt.title('Satisfaction Score by Accommodation Level')
plt.show()

## Question 3
Choose **three charts** from Task 3 and interpret them.

For each selected chart, write:

1. What relationship do you observe?
2. Is the pattern strong, moderate, weak, or unclear?
3. Does the relationship make business sense?
4. Would this relationship be useful for prediction, segmentation, or decision-making?

### Your answer

**Chart 1:**  

**Chart 2:**  

**Chart 3:**

---
# Task 4: Correlation matrix and heatmap

Correlation tells us the direction and strength of a **linear** relationship between numerical variables.

Important reminder:

> Correlation does not prove causation.

In [ ]:
# Encode simple binary variables for correlation analysis
analysis_df = df.copy()
analysis_df['GuideIncludedBinary'] = analysis_df['GuideIncluded'].map({'Yes': 1, 'No': 0})
analysis_df['HighValueBinary'] = analysis_df['HighValueBooking'].map({'Yes': 1, 'No': 0})
analysis_df['HighSatisfactionBinary'] = analysis_df['HighSatisfaction'].map({'Yes': 1, 'No': 0})

corr_cols = [
    'Travelers', 'LeadDays', 'TripDays', 'MaxAltitudeM', 'DifficultyScore',
    'PermitComplexityScore', 'AcclimatizationDays', 'GuideIncludedBinary',
    'SpendPerPersonUSD', 'TotalSpendUSD', 'SatisfactionScore',
    'RebookIntentScore', 'HighValueBinary', 'HighSatisfactionBinary'
]

corr = analysis_df[corr_cols].corr()
round(corr, 2)

In [ ]:
plt.figure(figsize=(13, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=.5)
plt.title('Correlation Heatmap: Nepal Tourism Booking Variables')
plt.show()

## Question 4
Using the correlation matrix or heatmap:

1. Identify **two feature-target relationships** with `TotalSpendUSD` or `HighValueBinary`.
2. Identify **two feature-feature relationships** that may indicate redundancy or multicollinearity.
3. Identify one correlation that is expected from domain knowledge.
4. Identify one correlation that needs caution or deeper interpretation.

### Your answer

1. Feature-target relationships:  

2. Feature-feature relationships:  

3. Expected correlation:  

4. Correlation needing caution:

---
# Task 5: Multicollinearity and VIF

Multicollinearity occurs when predictor variables are strongly related to one another.

This matters because highly related predictors may repeat the same signal and make model interpretation unstable.

In this section, we will use VIF to diagnose feature dependence.

In [ ]:


try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    import statsmodels.api as sm
    print('statsmodels is available')
except Exception as e:
    print('statsmodels is not available in this environment:', e)

In [ ]:
# VIF calculation for selected numerical predictors
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_features = [
    'Travelers', 'LeadDays', 'TripDays', 'MaxAltitudeM', 'DifficultyScore',
    'PermitComplexityScore', 'AcclimatizationDays', 'GuideIncludedBinary'
]

X = analysis_df[vif_features].copy()
X = sm.add_constant(X)

vif_table = pd.DataFrame({
    'Feature': X.columns,
    'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})

vif_table = vif_table[vif_table['Feature'] != 'const'].sort_values('VIF', ascending=False)
round(vif_table, 2)

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=vif_table, x='VIF', y='Feature')
plt.axvline(5, color='orange', linestyle='--', label='VIF = 5')
plt.axvline(10, color='red', linestyle='--', label='VIF = 10')
plt.title('Variance Inflation Factor by Feature')
plt.legend()
plt.show()

## Question 5
Interpret the VIF results.

1. Which variables have the highest VIF values?
2. What common tourism concept might these variables be measuring?
3. Which variables would you keep, remove, or combine if the goal were to build a simple and explainable model?
4. Would your answer change if the goal were only prediction accuracy?

### Your answer

---
A small model comparison

This section shows how a model can be affected by related features. The goal is not to build the best model. The goal is to see how feature choices change interpretation.

We will predict `SpendPerPersonUSD` using two feature sets:

- **Full feature set:** includes several related route-complexity variables.
- **Reduced feature set:** keeps fewer, more interpretable variables.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# Full and reduced features
full_features = ['Travelers','LeadDays','TripDays','MaxAltitudeM','DifficultyScore','PermitComplexityScore','AcclimatizationDays','GuideIncludedBinary']
reduced_features = ['Travelers','LeadDays','TripDays','GuideIncludedBinary']

y = analysis_df['SpendPerPersonUSD']

X_full = analysis_df[full_features]
X_reduced = analysis_df[reduced_features]

Xf_train, Xf_test, yf_train, yf_test = train_test_split(X_full, y, test_size=0.25, random_state=42)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_reduced, y, test_size=0.25, random_state=42)

model_full = LinearRegression().fit(Xf_train, yf_train)
model_reduced = LinearRegression().fit(Xr_train, yr_train)

pred_full = model_full.predict(Xf_test)
pred_reduced = model_reduced.predict(Xr_test)

print('Full model R2:', round(r2_score(yf_test, pred_full), 3))
print('Full model MAE:', round(mean_absolute_error(yf_test, pred_full), 2))
print('Reduced model R2:', round(r2_score(yr_test, pred_reduced), 3))
print('Reduced model MAE:', round(mean_absolute_error(yr_test, pred_reduced), 2))

coef_full = pd.DataFrame({'Feature': full_features, 'Coefficient': model_full.coef_}).sort_values('Coefficient', ascending=False)
coef_reduced = pd.DataFrame({'Feature': reduced_features, 'Coefficient': model_reduced.coef_}).sort_values('Coefficient', ascending=False)

print('\nFull model coefficients')
display(coef_full)
print('\nReduced model coefficients')
display(coef_reduced)

## Optional reflection
Compare the full and reduced models.

- Did the full model clearly outperform the reduced model?
- Which model is easier to explain to tourism managers?
- If prediction accuracy is similar, which model would you recommend and why?

### Your answer

---
# Final task: Model-readiness memo

Write a short memo of **250 to 350 words** to the tourism company.

Your memo should include:

1. Three important relationships discovered in the data.
2. One or two variables that may be redundant because of multicollinearity.
3. One recommendation for feature selection or feature engineering.
4. One warning about interpreting correlation.
5. Whether the dataset is ready for a first modelling experiment.

## Your memo:

